In [5]:
import boto3
import json

bedrock = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
bedrock_agent = boto3.client(service_name='bedrock-agent', region_name="us-east-1")

MODEL_ID = "amazon.nova-micro-v1:0"

In [6]:
temperature = .7

inference_config = {"temperature": temperature}

system_prompts = [{"text": "Você é um assistente de viagens virtual que sugere destinos com base nas preferências do usuário. "
                + "Retorne apenas o nome dos destinos e uma breve descrição."}]

messages = []

message_1 = {
    "role": "user",
    "content": [{"text": "Crie uma lista com 3 destinos de viagem."}]
}

messages.append(message_1)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

def print_response(response):
    model_response = response.get('output', {}).get('message', {}).get('content', [{}])[0].get('text', '')

    print("✈️ Destinos de viagem sugeridos para você:")
    print(model_response)

print_response(response)

✈️ Your suggested travel destinations:
1. **Kyoto, Japan**  
   Experience traditional Japanese culture with stunning temples, serene gardens, and the art of tea ceremonies.

2. **Banff, Canada**  
   Immerse yourself in the majestic Canadian Rockies, with opportunities for hiking, wildlife spotting, and relaxing hot springs.

3. **Marrakech, Morocco**  
   Discover vibrant souks, historic palaces, and the tranquil beauty of the Jardin Majorelle in this exotic Moroccan city.


In [7]:
message_2 = {
        "role": "user",
        "content": [{"text": "Sugira apenas destinos que estejam a no máximo um voo curto de distância."}]
}

messages.append(message_2)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **San Diego, California**
   - Brief Description: Enjoy beautiful beaches, world-class zoos, and vibrant cultural attractions along the stunning coastline.

2. **Seattle, Washington**
   - Brief Description: Experience lush greenery, iconic landmarks like the Space Needle, and rich coffee culture in this Pacific Northwest gem.

3. **Denver, Colorado**
   - Brief Description: Explore the Mile High City’s outdoor adventures, including mountains, parks, and unique urban experiences.


In [8]:
message_3 = {
        "role": "user",
        "content": [{"text": "Liste os destinos por proximidade de Belo Horizonte, MG, Brasil."}]
}

messages.append(message_3)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **São Paulo, SP, BR**  
   *Description:* São Paulo, Brazil's largest city, offers a vibrant mix of culture, art, and cuisine just a 1-hour flight from Belo Horizonte.

2. **Rio de Janeiro, RJ, BR**  
   *Description:* Known for its iconic landmarks like Christ the Redeemer and Copacabana Beach, Rio de Janeiro is accessible within a 2-hour flight from Belo Horizonte.

3. **Buenos Aires, AR**  
   *Description:* The lively capital of Argentina, featuring tango, vibrant nightlife, and delicious cuisine, is reachable within a 2-hour flight from Belo Horizonte.


In [9]:
try:
    response = bedrock_agent.create_prompt(
        name="Travel-Agent-Prompt",
        description="Verifica se todas as informações da viagem foram fornecidas.",
        variants=[
            { 
                "name": "Variant1",
                "modelId": MODEL_ID,
                "templateType": "CHAT",
                "inferenceConfiguration": {
                    "text": {
                        "temperature": 0.4
                    }
                },
                "templateConfiguration": { 
                    "chat": {
                        'system': [ 
                            {
                                "text": """Você é um agente de viagens avaliando pedidos de roteiros personalizados.
                                Leia a mensagem com atenção e responda SIM ou NÃO para as perguntas de triagem abaixo.
                                Seja rigoroso: se algum detalhe estiver faltando ou não estiver claro, responda NÃO.

                                A) O destino está claramente informado?
                                B) As datas da viagem estão dentro de um prazo razoável (nem de última hora, nem para daqui a mais de um ano)?
                                C) O pedido evita atividades de alto risco ou restritas (ex.: esportes radicais, viagens para locais isolados)?
                                D) Há alguma menção a passaporte válido ou documentação de viagem?
                                E) Há informação suficiente para dar retorno com uma proposta de roteiro?"""
                            }
                        ],
                        'messages': [{
                            'role': 'user',
                            'content': [ 
                                {
                                    'text': "Pedido de viagem: {{event_request}}"
                                }
                            ]
                        }],
                        'inputVariables' : [
                            { 'name' : 'event_request'}
                        ]
                    }
                }
        }]
    )
    print("Criado!")
    prompt_arn = response.get("arn")
except bedrock.exceptions.ConflictException as e:
    print("Já existe!")
    response = bedrock.list_prompts()
    prompt = next((prompt for prompt in response['promptSummaries'] if prompt['name'] == "TripBooker_xyz"), None)
    prompt_arn = prompt['arn']

prompt_arn

Created!


'arn:aws:bedrock:us-east-1:<ACCOUNT_ID>:prompt/Q5GRIL8PZD'

In [10]:
response = bedrock.converse(
    modelId=prompt_arn,
    promptVariables={
        'event_request': {
            'text': """
                Olá! Estou planejando uma viagem para a Itália com meu parceiro e gostaria de ajuda para organizar o roteiro. Queremos viajar entre 10 e 20 de setembro deste ano, de preferência chegando por Roma e passando também alguns dias em Florença e Veneza. Adoraríamos recomendações de passeios, pontos culturais e bons restaurantes locais. Não temos interesse em nada arriscado, como paraquedismo ou trilhas em locais isolados — só queremos uma experiência tranquila e enriquecedora. Nós dois temos passaporte válido. Me diga quais outros detalhes você precisa!
                """
        }
    },
)
print(response['output']['message']['content'][0]['text'])

YES

A) Is the destination clearly stated?
YES. The destination is clearly stated as Italy.

B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
YES. The travel dates are within a reasonable range, specifically between September 10-20 this year.

C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
YES. The request explicitly states a preference for non-risky activities, avoiding extreme sports and off-grid travel.

D) Is there any mention of a valid passport or travel documentation?
YES. Both travelers have valid passports.

E) Is there enough information to follow up with a proposed itinerary?
YES. There is sufficient detail provided to create a proposed itinerary, including the destination, travel dates, preferred cities (Rome, Florence, and Venice), and a desire for cultural experiences and local dining recommendations.
